# Food–Microbiome Linkage

Goal of this notebook: link the two project datasets (cFMD and dietstudy) by
building a new unified key column (`food_category_unified`), since no shared
field exists between them natively.

## Step 1: Extract unique values from both sides

Before building any mapping, we need to know exactly what we're working with:
- All unique (L1, L2) USDA taxonomy pairs that actually appear in the food diaries
- All unique macrocategory/category/subtype values in cFMD, including sample
  counts per value (to identify where real data overlap exists)

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)

BASE = Path("dietstudy_analyses-master")
FOOD_DIR = BASE / "data" / "procrustes" / "data_username_1day"

# ---------- Dataset 2: extract L1/L2 from the USDA taxonomy strings ----------
food_files = sorted(FOOD_DIR.glob("*_food.txt"))

usda_rows = []
for fp in food_files:
    df = pd.read_csv(fp, sep="\t", usecols=["#FoodID", "taxonomy"])
    usda_rows.append(df)

usda_all = pd.concat(usda_rows, ignore_index=True).drop_duplicates(subset="#FoodID")

def extract_level(taxonomy, level_idx):
    parts = str(taxonomy).split(";")
    if level_idx < len(parts):
        return parts[level_idx].replace(f"L{level_idx+1}_", "").strip()
    return "Unknown"

usda_all["L1"] = usda_all["taxonomy"].apply(lambda t: extract_level(t, 0))
usda_all["L2"] = usda_all["taxonomy"].apply(lambda t: extract_level(t, 1))

usda_l1_l2_counts = (
    usda_all.groupby(["L1", "L2"])["#FoodID"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="NumUniqueFoods")
)

print(f"Total unique food items in diary taxonomy: {usda_all['#FoodID'].nunique()}")
print(f"Total unique (L1, L2) pairs: {len(usda_l1_l2_counts)}")
print()
usda_l1_l2_counts

Total unique food items in diary taxonomy: 1452
Total unique (L1, L2) pairs: 45



,L1,L2,NumUniqueFoods
0,Vegetables,Other_vegetables,190
1,Grain_Product,Grain_mixtures_frozen_plate_meals_soups,132
2,Grain_Product,Cakes_cookies_pies_pastries_bars,81
3,Meat_Poultry_Fish_and_Mixtures,Meatpoultry_fish_with_nonmeat,71
4,Grain_Product,Yeast,64
5,Sugars_Sweets_and_Beverages,Nonalcoholic_beverages,59
6,Sugars_Sweets_and_Beverages,Sugars_and_sweets,57
7,Meat_Poultry_Fish_and_Mixtures,Poultry,42
8,Fruits,Other_fruits,42
9,Vegetables,White_potatoes_and_Puerto_Rican_starchy_vegetables,41


In [2]:
# ---------- Dataset 1: unique cFMD category values + sample counts ----------
cfmd_metadata = pd.read_csv("master_metadata.tsv", sep="\t")

cfmd_category_counts = (
    cfmd_metadata.groupby(["macrocategory", "category"])
    .size()
    .sort_values(ascending=False)
    .reset_index(name="NumSamples")
)

print(f"Total cFMD samples: {len(cfmd_metadata)}")
print(f"Total unique (macrocategory, category) pairs: {len(cfmd_category_counts)}")
print()
cfmd_category_counts

Total cFMD samples: 3441
Total unique (macrocategory, category) pairs: 15



,macrocategory,category,NumSamples
0,food,dairy,2296
1,food,fermented_beverages,435
2,food,fermented_meat,156
3,food,meat,119
4,food,fish,91
5,food,fermented_grains,83
6,food,fermented_seeds,65
7,food,fermented_fruits_and_vegetables,52
8,food,alcohol,48
9,food,fruits_and_vegetables,25


## Step 1b: Inspect subtype breakdown within the largest cFMD categories

Before committing to a mapping, we check whether the largest categories
(`dairy`, `fermented_fruits_and_vegetables`, `fermented_grains`) are
homogeneous (all genuinely fermented) or mixed with non-fermented items,
since this directly affects whether matching them to USDA categories is valid.

In [3]:
for cat in ["dairy", "fermented_fruits_and_vegetables", "fermented_grains", "fermented_legumes"]:
    subset = cfmd_metadata[cfmd_metadata["category"] == cat]
    print(f"=== category = {cat} (n={len(subset)}) ===")
    print(subset["subtype"].value_counts().head(15))
    print()

=== category = dairy (n=2296) ===
subtype
milk_kefir                      285
raw_milk_cheese                 128
Cheddar                         102
Cheese_1_before_ripening         97
cheese_2_before_ripening         96
cheese_2_after_ripening          95
Cheese_1_after_ripening          94
whey                             84
raw_milk                         70
Caciocavallo                     66
surface_ripened_soft_cheeses     56
parmigiano_reggiano              53
brine                            53
grana                            50
mozzarella                       49
Name: count, dtype: int64

=== category = fermented_fruits_and_vegetables (n=52) ===
subtype
mixed_vegetable_pickle                         12
green_papaya_pickle                             3
white_onion_pickle                              3
vitis_vinifera_l_cv_corvina_grape_withering     2
carrot_kimchi                                   1
garlic_kraut                                    1
brussels_sprout_kimchi   

## Step 1c: Search USDA food names for fermentation-relevant keywords

Rather than matching whole L1/L2 categories (which are too broad and mostly
non-fermented), we search the actual food item names and L3-L5 levels for
keywords that indicate genuine fermentation, to find the narrow subset of
the diary that has a real fermented-food counterpart in cFMD.

In [4]:
FERMENT_KEYWORDS = [
    "pickle", "kimchi", "sauerkraut", "kraut", "fermented",
    "sourdough", "kefir", "kombucha", "miso", "yogurt", "yoghurt"
]

pattern = "|".join(FERMENT_KEYWORDS)
ferment_matches = usda_all[
    usda_all["taxonomy"].str.contains(pattern, case=False, na=False)
    | usda_all["#FoodID"].str.contains(pattern, case=False, na=False)
]

print(f"USDA food items matching fermentation keywords: {len(ferment_matches)}")
print()
ferment_matches[["#FoodID", "L1", "L2"]].sort_values("L1")

USDA food items matching fermentation keywords: 30



,#FoodID,L1,L2
1213,Soybean soup miso broth,Dry_Beans_Peas_Other_Legumes_Nuts_and_Seeds,Legumes
1436,Yogurt dressing,Fats_Oils_and_Salad_Dressings,Salad_dressings
836,Nature Valley Chewy Granola Bar with Yogurt Coating,Grain_Product,Cakes_cookies_pies_pastries_bars
1449,Yogurt vanilla lemon or coffee flavor whole milk,Milk_and_Milk_Products,Milks_and_milk_drinks
1447,Yogurt vanilla lemon maple or coffee flavor lowfat milk,Milk_and_Milk_Products,Milks_and_milk_drinks
1446,Yogurt plain whole milk,Milk_and_Milk_Products,Milks_and_milk_drinks
1445,Yogurt plain nonfat milk,Milk_and_Milk_Products,Milks_and_milk_drinks
1444,Yogurt plain lowfat milk,Milk_and_Milk_Products,Milks_and_milk_drinks
1443,Yogurt fruit variety whole milk,Milk_and_Milk_Products,Milks_and_milk_drinks
1442,Yogurt fruit variety nonfat milk sweetened with low calorie sweetener,Milk_and_Milk_Products,Milks_and_milk_drinks


## Step 1d: Check for fermented-meat matches (salami, sausage, cured meats)

cFMD's `fermented_meat` category (n=156) may have a counterpart in the diary's
sausage/lunchmeat items. We check explicitly since "sausage" does not contain
any of the previous fermentation keywords.

In [5]:
MEAT_KEYWORDS = ["salami", "sausage", "pepperoni", "prosciutto", "cured", "bologna", "lunchmeat", "ham"]

pattern_meat = "|".join(MEAT_KEYWORDS)
meat_matches = usda_all[
    usda_all["#FoodID"].str.contains(pattern_meat, case=False, na=False)
]

print(f"USDA food items matching meat-curing keywords: {len(meat_matches)}")
print()
meat_matches[["#FoodID", "L1", "L2"]].sort_values("#FoodID")

USDA food items matching meat-curing keywords: 38



,#FoodID,L1,L2
108,Bologna turkey,Meat_Poultry_Fish_and_Mixtures,Organ_meats_sausages_and_lunchmeats
491,Crackers graham,Grain_Product,Crackers_and_salty_snacks_from_grain
539,Deer bologna,Meat_Poultry_Fish_and_Mixtures,Lamb_veal_game_other
552,Egg cheese and ham on bagel,Eggs,Egg_mixtures
553,Egg cheese and sausage on biscuit,Eggs,Egg_mixtures
662,Ham fried NS as to fat eaten,Meat_Poultry_Fish_and_Mixtures,Pork
663,Ham or pork noodles and vegetables excluding carrots broccoli and dark green...,Meat_Poultry_Fish_and_Mixtures,Meatpoultry_fish_with_nonmeat
664,Ham or pork salad,Meat_Poultry_Fish_and_Mixtures,Meatpoultry_fish_with_nonmeat
665,Ham or pork with barbecue sauce mixture,Meat_Poultry_Fish_and_Mixtures,Meatpoultry_fish_with_nonmeat
666,Ham prosciutto,Meat_Poultry_Fish_and_Mixtures,Pork


## Step 1e: Inspect subtype breakdown of cFMD fermented_meat

Before matching salami/pepperoni-type items, we verify what cFMD's
fermented_meat category actually contains, to avoid repeating the mistake
seen with fermented_grains (mostly non-relevant ethnic products).

In [6]:
subset = cfmd_metadata[cfmd_metadata["category"] == "fermented_meat"]
print(f"=== category = fermented_meat (n={len(subset)}) ===")
print(subset["subtype"].value_counts().head(20))

=== category = fermented_meat (n=156) ===
subtype
cured_meat_after_ripening                                           16
felino-type_sausages                                                16
pig_sausage                                                         15
sausage_after_ripening                                              14
salame_felino                                                       10
finocchiona                                                         10
salame_napoli                                                        9
fresh_meat_after_ripening                                            8
calibraton_sausage_containing_cattle_pig_chicken_and_turkey_meat     8
salame_ungherese                                                     7
salame_felino_IGP                                                    6
cacciatore_italiano                                                  6
calibraton_sausage_containing_cattle_pig_sheep_and_horse_meat        4
salame_piacentino          

## Step 2: Build the explicit mapping dictionaries and unified key

Based on the inspection above, we define two lookup tables that map each
side to a shared `food_category_unified` key. Categories with no validated
counterpart (fermented_grains, fermented_legumes, fermented_fish,
fermented_tubers_and_roots) are intentionally excluded — see documentation
for rationale.

In [9]:
# ---------- cFMD side: category -> unified key ----------
# Note: "probiotics" was checked and has zero counterpart in the diary
# (no USDA food item matched it) - excluded to avoid dead code.
CFMD_CATEGORY_MAP = {
    "dairy": "dairy_fermented",
    "fermented_beverages": "beverage_alcoholic_fermented",
    "alcohol": "beverage_alcoholic_fermented",
    "fermented_meat": "meat_cured_fermented",
    "fermented_fruits_and_vegetables": "vegetable_fermented",
}

cfmd_metadata["food_category_unified"] = cfmd_metadata["category"].map(CFMD_CATEGORY_MAP)

print("cFMD samples retained after mapping:")
print(cfmd_metadata["food_category_unified"].value_counts(dropna=False))

cFMD samples retained after mapping:
food_category_unified
dairy_fermented                 2296
beverage_alcoholic_fermented     483
NaN                              454
meat_cured_fermented             156
vegetable_fermented               52
Name: count, dtype: int64


In [8]:
# ---------- USDA side: explicit item-level allow-list per unified key ----------

DAIRY_FERMENTED_ITEMS = [
    # all Cheeses (L2) are fermented dairy products
    "L2_Cheeses",
]
DAIRY_FERMENTED_FOODIDS = [
    "Yogurt vanilla lemon or coffee flavor whole milk",
    "Yogurt vanilla lemon maple or coffee flavor lowfat milk",
    "Yogurt vanilla lemon maple or coffee flavor nonfat milk",
    "Yogurt plain whole milk", "Yogurt plain nonfat milk", "Yogurt plain lowfat milk",
    "Yogurt fruit variety whole milk",
    "Yogurt fruit variety nonfat milk sweetened with low calorie sweetener",
    "Yogurt fruit variety nonfat milk",
    "Yogurt fruit variety lowfat milk sweetened with low calorie sweetener",
    "Yogurt fruit variety lowfat milk",
    "Yogurt frozen NS as to flavor NS as to type of milk",
    "Yogurt frozen flavors other than chocolate lowfat milk",
]  # explicitly excludes "Yogurt dressing" and "...Yogurt Coating" (not real yogurt exposure)

ALCOHOL_FERMENTED_L2 = ["L2_Alcoholic_beverages"]

MEAT_CURED_FOODIDS = [
    "Salami NFS", "Pepperoni", "Italian sausage", "Polish sausage", "Turkey salami",
]  # explicitly excludes Bologna/Ham/Bacon/Pork sausage/Turkey sausage (smoked/cooked, not fermented-cured)

VEGETABLE_FERMENTED_FOODIDS = [
    "Beans string green pickled", "Peppers pickled", "Olives NFS", "Olives green",
    "Olives black", "Cucumber pickles relish", "Cucumber pickles dill reduced salt",
    "Cucumber pickles dill", "Cabbage red pickled", "Beets pickled",
    "Radishes pickled Hawaiian style", "Sauerkraut cooked NS as to fat added in cooking",
]  # excludes "Mustard" and "Hot pepper sauce" (condiments, not whole fermented vegetables)


def assign_unified_category(row):
    food_id = row["#FoodID"]
    if food_id in DAIRY_FERMENTED_FOODIDS or row["L2"] == "Cheeses":
        return "dairy_fermented"
    if row["L2"] == "Alcoholic_beverages":
        return "beverage_alcoholic_fermented"
    if food_id in MEAT_CURED_FOODIDS:
        return "meat_cured_fermented"
    if food_id in VEGETABLE_FERMENTED_FOODIDS:
        return "vegetable_fermented"
    return None


usda_all["food_category_unified"] = usda_all.apply(assign_unified_category, axis=1)

print("USDA diary items retained after mapping:")
print(usda_all["food_category_unified"].value_counts(dropna=False))

USDA diary items retained after mapping:
food_category_unified
None                            1366
dairy_fermented                   50
beverage_alcoholic_fermented      19
vegetable_fermented               12
meat_cured_fermented               5
Name: count, dtype: int64


## Step 3: Compute a microbial signature for each food category

For each of the 4 validated `food_category_unified` groups, we summarize the
microbial richness (number of taxa present) and Shannon diversity across all
cFMD samples belonging to that category. This produces one row per category -
the "microbial fingerprint" that will later be joined to the diary side.

Richness and diversity are computed per-sample first, then averaged across
samples within each category (not the reverse), to avoid letting samples
with many zero-abundance taxa distort the category-level mean.

In [12]:
from scipy.stats import entropy

cfmd_tax = pd.read_csv("master_taxonomic_profiles.tsv", sep="\t", index_col=0)

# The raw file has mixed-type columns (numbers stored as strings in some
# columns) - force everything to numeric, turning any non-numeric value into 0.
cfmd_tax = cfmd_tax.apply(pd.to_numeric, errors="coerce").fillna(0)

# Restrict to samples that have a valid unified category
valid_samples = cfmd_metadata.dropna(subset=["food_category_unified"])

if "combined_id" not in valid_samples.columns:
    valid_samples = valid_samples.copy()
    valid_samples["combined_id"] = (
        valid_samples["dataset_name"] + "__" + valid_samples["sample_id"].astype(str)
    )

valid_samples = valid_samples[valid_samples["combined_id"].isin(cfmd_tax.columns)]
print(f"cFMD samples with valid category AND present in taxonomic table: {len(valid_samples)}")

per_sample_records = []
for _, row in valid_samples.iterrows():
    abundances = cfmd_tax[row["combined_id"]].to_numpy()
    abundances = abundances[abundances > 0]
    if len(abundances) == 0:
        continue
    per_sample_records.append({
        "combined_id": row["combined_id"],
        "food_category_unified": row["food_category_unified"],
        "Richness": len(abundances),
        "ShannonDiversity": entropy(abundances),
    })

per_sample_df = pd.DataFrame(per_sample_records)

category_signature = (
    per_sample_df.groupby("food_category_unified")[["Richness", "ShannonDiversity"]]
    .mean()
    .reset_index()
)
category_signature["NumSamplesUsed"] = (
    per_sample_df.groupby("food_category_unified").size().values
)

print()
category_signature

C:\Users\winte\AppData\Local\Temp\ipykernel_45580\1452544241.py:3: DtypeWarning: Columns (823) have mixed types. Specify dtype option on import or set low_memory=False.
  cfmd_tax = pd.read_csv("master_taxonomic_profiles.tsv", sep="\t", index_col=0)


cFMD samples with valid category AND present in taxonomic table: 2952



,food_category_unified,Richness,ShannonDiversity,NumSamplesUsed
0,beverage_alcoholic_fermented,17.856846,1.297330,482
1,dairy_fermented,25.396118,1.063938,2267
2,meat_cured_fermented,37.410596,1.210545,151
3,vegetable_fermented,31.480769,1.727883,52
